In [167]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from scipy.interpolate import griddata

In [168]:
df = pd.read_csv('results_splice_2024-05-17_11-41-29.csv')
# df = pd.read_csv('AUBC_test.csv')

df.columns.values[0] = 'iterations'

print(df.shape)
df.head()

(200, 4)


,iterations,L2_value,drop_value,loss
0,0,0.007138,0.161918,0.496413
1,1,0.000006,0.039467,0.475625
2,2,0.012502,0.272261,0.467298
3,3,0.000015,0.171045,0.471171
4,4,0.002253,0.120208,0.486676


In [169]:
# Calculate the interval between each instance
df['interval'] = df['iterations'].diff().fillna(0)

# Calculate the trapezoidal area
df.loc[:, 'trapezoidal_area'] = 0.5 * (df['loss'] + df['loss'].shift(1, fill_value=0)) * df['interval']

df['cumulative_area'] = df['trapezoidal_area'].cumsum()

df['AUBC'] = df['cumulative_area'] / (df['cumulative_area'].max() * df.shape[0])

df['AUBC_totals'] = df['AUBC'].cumsum()

df

,iterations,L2_value,drop_value,loss,interval,trapezoidal_area,cumulative_area,AUBC,AUBC_totals
0,0,0.007138,0.161918,0.496413,0.0,0.000000,0.000000,0.000000,0.000000
1,1,0.000006,0.039467,0.475625,1.0,0.486019,0.486019,0.000027,0.000027
2,2,0.012502,0.272261,0.467298,1.0,0.471461,0.957480,0.000053,0.000080
3,3,0.000015,0.171045,0.471171,1.0,0.469235,1.426715,0.000079,0.000158
4,4,0.002253,0.120208,0.486676,1.0,0.478924,1.905638,0.000105,0.000263
...,...,...,...,...,...,...,...,...,...
195,195,0.000010,0.053120,0.484634,1.0,0.476250,88.893164,0.004902,0.481827
196,196,0.000021,0.443161,0.450527,1.0,0.467580,89.360744,0.004928,0.486755
197,197,0.000003,0.126986,0.432414,1.0,0.441470,89.802215,0.004952,0.491707
198,198,0.000611,0.228616,0.422617,1.0,0.427516,90.229730,0.004976,0.496682


In [170]:
def plot_3d_scatter_surface(df, x_variable, y_variables, output_variable, surface=True, highlight=False):
    name = ['L2', 'Dropout']
    
    for i, y_var in enumerate(y_variables):
        print(f'{name[i]}: Min value = {df[y_var].min()}, Max value = {df[y_var].max()}, Min Loss = {df[output_variable].min()}')
        # Find the index of the lowest output variable value if highlight is True
        lowest_index = df[output_variable].idxmin() if highlight else None
        
        # Create a scatter plot
        scatter = go.Scatter3d(
            x=df[x_variable],
            y=df[y_var],
            z=df[output_variable],
            mode='markers',
            marker=dict(
                size=[15 if i == lowest_index else 5 for i in range(len(df))],  # Highlighted point is bigger,
                color=['red' if i == lowest_index else 'grey' for i in range(len(df))] if highlight else 'grey',  # Highlight the lowest value if highlight is True
                colorscale='Viridis',  # Choose a color scale
                opacity=0.8,
            ),
            name=output_variable
        )

        # Create surface data
        x_data = df[x_variable]
        y_data = df[y_var]
        z_data = df[output_variable]

        x_range = np.linspace(min(x_data), max(x_data), 100)
        y_range = np.linspace(min(y_data), max(y_data), 100)
        X, Y = np.meshgrid(x_range, y_range)
        Z = griddata((x_data, y_data), z_data, (X, Y), method='cubic')

        if surface:
            # Create a 3D surface
            surface = go.Surface(
                x=X,
                y=Y,
                z=Z,
                colorscale='Jet',  # Choose a color scale
                opacity=0.6
            )

            # Create the figure
            fig = go.Figure(data=[scatter, surface])

        else:
            fig = go.Figure(data=[scatter])

        # Update layout
        fig.update_layout(
            scene=dict(
                xaxis_title=x_variable,
                yaxis_title=y_var,
                zaxis_title=output_variable
            ),
            width=800,  # adjust width
            height=600,  # adjust height
            title=f'Mean Loss for different values of {name[i]} regularization'
        )

        # Show the plot
        fig.show()

In [171]:
plot_3d_scatter_surface(df, 'iterations', ['L2_value', 'drop_value'], 'loss', surface=False, highlight=True)

L2: Min value = 1.0133468884667797e-06, Max value = 0.0180332369939233, Min Loss = 0.4184151023626327


Dropout: Min value = 0.0174179051927751, Max value = 0.4984902408019843, Min Loss = 0.4184151023626327


In [172]:
def plot_2d_scatter(df, x_column, output_column, highlight=False):

    # Find the index of the lowest output variable value if highlight is True
    lowest_index = df[output_column].idxmin() if highlight else None
    
    # Create a scatter plot
    scatter = go.Scatter(
        x=df[x_column],
        y=df[output_column],
        mode='lines+markers',
    
        marker=dict(
            size=[15 if i == lowest_index else 5 for i in range(len(df))],  # Highlighted point is bigger
            color=['red' if i == lowest_index else 'blue' for i in range(len(df))] if highlight else 'blue',  # Highlight the lowest value if highlight is True
            opacity=0.8,
        ),
        name=output_column
    )

    # Create the figure
    fig = go.Figure(data=[scatter])

    # Update layout
    fig.update_layout(
        xaxis_title=x_column,
        yaxis_title=output_column,
        # yaxis=dict(tickformat='e'),
        width=800,  # adjust width
        height=600,  # adjust height
        title=f'{output_column} as a function of {x_column}'
    )

    # Show the plot
    fig.show()

In [173]:
plot_2d_scatter(df, 'iterations', 'loss', highlight=True)

In [174]:
plot_2d_scatter(df, 'iterations', 'L2_value', highlight=True)


In [175]:
plot_2d_scatter(df, 'iterations', 'drop_value', highlight=True)

In [176]:
print(f"Area under the curve (AUC): {df['AUBC_totals'].iloc[-1]:.3f}")

plot_2d_scatter(df, 'iterations', 'AUBC', highlight=False)

Area under the curve (AUC): 0.502
